In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [2]:
a = 0.2
m = 0.07
v = 182.5

Nx, Ny = 40, 40
dx = dy = 1
T = 20

w0 = np.ones((Nx, Ny)) * a
n0 = 0.1 * np.random.rand(Nx, Ny)

In [3]:
# Flatten para usar com solve_ivp
y0 = np.concatenate([w0.ravel(), n0.ravel()])

In [4]:
# Operadores espaciais
def laplacian(Z):
    return (
        np.roll(Z, -1, axis=0) + np.roll(Z, 1, axis=0) +
        np.roll(Z, -1, axis=1) + np.roll(Z, 1, axis=1) -
        4 * Z
    ) / dx**2

def advect_x(Z):
    return (Z - np.roll(Z, 1, axis=0)) / dx

# RHS
def rhs(w, n):
    dw = a - w - w*n**2 + v * advect_x(w)
    dn = w*n**2 - m*n + laplacian(n)
    return dw, dn

In [5]:
def rhs_flat_python(t, y):
    w = y[:Nx*Ny].reshape(Nx, Ny)
    n = y[Nx*Ny:].reshape(Nx, Ny)
    dw, dn = rhs(w, n)  # rhs normal, sem Numba
    return np.concatenate([dw.ravel(), dn.ravel()])

In [6]:
def plot_solution_no_pause(sol, Nx, Ny, snapshot_interval=1.0):
    snapshots = []
    snapshot_times = []
    last_snapshot_time = 0.0

    # salva snapshots
    for idx, t in enumerate(sol.t):
        if t - last_snapshot_time >= snapshot_interval or idx == len(sol.t)-1:
            y = sol.y[:, idx]
            n = y[Nx*Ny:].reshape(Nx, Ny)
            snapshots.append(n.copy())
            snapshot_times.append(t)
            last_snapshot_time = t

    # plota todos de uma vez
    n_snapshots = len(snapshots)
    cols = min(5, n_snapshots)
    rows = (n_snapshots + cols - 1) // cols

    plt.figure(figsize=(4*cols, 4*rows))
    for i, (n_snap, t_snap) in enumerate(zip(snapshots, snapshot_times)):
        ax = plt.subplot(rows, cols, i+1)
        im = ax.imshow(n_snap, origin='lower', cmap='Greens')
        ax.set_title(f'T = {t_snap:.2f}')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [7]:
# Integração com RK45 adaptativo
sol = solve_ivp(
    rhs_flat_python,
    t_span=(0, T),
    y0=y0,
    method='BDF',
    rtol=1e-3,
    atol=1e-6,
    max_step=0.1,
    vectorized=False
)

# Plot
plot_solution_no_pause(sol, Nx, Ny, snapshot_interval=5.0)

KeyboardInterrupt: 